# ACC theta PAC vs. Haufe-scaled electrode reactivation (classical PAC)

For each subject:

1. Load cue-period **ACC HFA** `(n_trials, n_acc_elec, n_t)` and the subject's **encoding Haufe**
   vector `w_subj` on the same electrodes. Build **per-electrode reactivation**
   `r[t, e, t] = cue_hfa[t, e, t] * w_subj[e]` so Haufe weights scale each channel's amplitude
   independently (contrast changes `w` and therefore each electrode's amplitude).
2. Extract **ACC theta phase** (`4-8` Hz, Hilbert) at the same cue-time samples for every ACC
   electrode (`extract_theta_phase_from_cue_epochs`), on the HFA time grid.
3. For each electrode `e`, run tensorpac **Tort MI** PAC with **phase permutation surrogates**
   (`Pac(idpac=(2, 1, 0))`) on 3D inputs `(1, n_trials, n_t)` — same-electrode phase vs
   Haufe-scaled HFA amplitude — **no Gaussian copula** (`idpac[0] != 6`). Per-trial z uses
   surrogate mean/std (same spirit as the previous pipeline, without copula normalization).
4. Optionally restrict to **positive RPE trials** only (`rpe_cat == 'pos'`, within-subject
   terciles via `build_rpe_trial_map` on `cue_on`) so cue-period signals align with the
   positive-RPE encoding contrast; hit/miss summaries are then **within that subset** (often
   fewer trials per subject).
5. Aggregate per-trial z across electrodes (`mean` over `e`) for hit/miss summaries; full
   `(n_trials, n_elec)` arrays are stored in the NPZ for diagnostics.

Outputs live under `analysis_outputs/acc_thetapac_reactivation/<SELECTED_CONTRAST>/`.

**Cache schema marker**: `PAC_PRIMITIVE = 'haufe_reactivation_per_elec_phase_amp_pac_posrpe_v1'` plus
`trial_filter` (`pos_rpe_tercile` vs `all_trials`) in every per-subject NPZ.


In [ ]:
%matplotlib inline
%reload_ext autoreload
%autoreload 2

import os
import sys
import warnings
import inspect
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple

# Force local patched tensorpac before any tensorpac imports.
TP_LOCAL_ROOT = '/Users/salmanqasim/Documents/GitRepos/tensorpac'
if TP_LOCAL_ROOT in sys.path:
    sys.path.remove(TP_LOCAL_ROOT)
sys.path.insert(0, TP_LOCAL_ROOT)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from joblib import Parallel, delayed
from scipy.stats import ttest_rel, ttest_1samp, wilcoxon
from tqdm.auto import tqdm

import mne
import tensorpac
from tensorpac import Pac

from plot_config import set_plot_style
from mv_regression_utils import (
    Subject,
    _get_smoothed_event_signal,
    extract_theta_phase_from_cue_epochs,
)
from mv_granger_utils import build_hit_miss_trial_map, build_rpe_trial_map

warnings.filterwarnings('ignore')
set_plot_style()
sns.set_style('white')

print('tensorpac.__version__ =', getattr(tensorpac, '__version__', 'unknown'))
print('tensorpac.__file__    =', tensorpac.__file__)
assert '/Users/salmanqasim/Documents/GitRepos/tensorpac/tensorpac/' in tensorpac.__file__.replace('\\', '/'), (
    'Notebook is not using local patched tensorpac. Restart kernel and run this cell first.'
)


In [ ]:
# Sanity: tensorpac Pac.fit available (per-electrode PAC uses Tort MI idpac, not copula).
src = inspect.getsource(Pac.fit)
assert 'def fit' in src
print('tensorpac Pac.fit available (ok for idpac=(2, 1, 0) per-electrode PAC).')


In [ ]:
# -----------------------------
# Paths
# -----------------------------
base_dir = '/Volumes/T7 Shield'
load_dir = f'{base_dir}/work/qasims01/MemoryBanditData/EMU'

HAUFE_PATTERNS_CSV: Path = Path(
    '/Volumes/T7 Shield/scratch/MemoryBandit/MVPA/'
    'decoding_feedback_posthoc_patterns/ACC_rpe_category_hfa_200_posthoc_patterns.csv'
)

# -----------------------------
# Active contrast
# -----------------------------
SELECTED_CONTRAST: str = 'positive_vs_neutral'    # or 'positive_vs_negative'
HAUFE_WEIGHT_COLUMN: str = 'PatternValue'         # 'PatternValue' (Haufe) or 'WeightValue' (raw)

analysis_output_root = Path(
    '/Users/salmanqasim/Documents/GitRepos/MemoryBandit_EMU/Notebooks/'
    'analysis_outputs/acc_thetapac_reactivation'
) / SELECTED_CONTRAST
per_subject_output_root = analysis_output_root / 'per_subject'
figure_output_root = analysis_output_root / 'figures'
for d in (per_subject_output_root, figure_output_root):
    d.mkdir(parents=True, exist_ok=True)

# -----------------------------
# HFA / theta config (kept aligned with MemoryBandit-ACC_HPC_PhaseRSA.ipynb)
# -----------------------------
ACC_ROI: str = 'ACC'
EVENT_CUE: str = 'cue_on'

CUE_WINDOW: Tuple[float, float] = (0.0, 1.0)
THETA_BAND: Tuple[float, float] = (4.0, 8.0)
HFA_BAND: Tuple[float, float] = (70.0, 201.0)
DOWNSAMPLE_FACTOR: int = 5
WIN_MS: int = 200
HFA_SMOOTH_METHOD: str = 'movmean'
HFA_ARTIFACT_ABS_THRESH: float = 10.0

# -----------------------------
# PAC config
# -----------------------------
N_PAC_PERM: int = 250          # tensorpac surrogates per subject (swap phase across trials)
PAC_RANDOM_STATE: int = 7

# -----------------------------
# Subject / electrode eligibility
# -----------------------------
MIN_TRIALS: int = 10
MIN_TRIALS_PER_COND: int = 8
# When restricting to positive-RPE (tercile) trials, hit/miss cells are sparser — use a lower floor.
MIN_TRIALS_PER_COND_POS_RPE: int = 5
MIN_PATTERN_ELECTRODES: int = 2

# Restrict cue epochs to within-subject tercile `rpe_cat == 'pos'` (see `build_rpe_trial_map`).
RESTRICT_TRIALS_TO_POS_RPE: bool = True


def min_trials_per_cond_effective() -> int:
    return MIN_TRIALS_PER_COND_POS_RPE if RESTRICT_TRIALS_TO_POS_RPE else MIN_TRIALS_PER_COND


TRIAL_FILTER_LABEL: str = 'pos_rpe_tercile' if RESTRICT_TRIALS_TO_POS_RPE else 'all_trials'

# -----------------------------
# Parallelism
# -----------------------------
N_JOBS: int = -1
JOBLIB_BACKEND: str = 'loky'

# -----------------------------
# Cache schema marker
# -----------------------------
PAC_PRIMITIVE: str = 'haufe_reactivation_per_elec_phase_amp_pac_posrpe_v1'

print('Output roots:')
for d in (per_subject_output_root, figure_output_root):
    print(f'  {d}')
print(f'HAUFE_PATTERNS_CSV    : {HAUFE_PATTERNS_CSV}')
print(f'SELECTED_CONTRAST     : {SELECTED_CONTRAST}')
print(f'HAUFE_WEIGHT_COLUMN   : {HAUFE_WEIGHT_COLUMN}')
print(f'CUE_WINDOW            : {CUE_WINDOW}')
print(f'THETA_BAND            : {THETA_BAND}')
print(f'HFA_BAND              : {HFA_BAND}')
print(f'N_PAC_PERM            : {N_PAC_PERM}')
print(f'MIN_PATTERN_ELECTRODES: {MIN_PATTERN_ELECTRODES}')
print(f'MIN_TRIALS_PER_COND   : {MIN_TRIALS_PER_COND}')
print(f'MIN_TRIALS_PER_COND_POS_RPE: {MIN_TRIALS_PER_COND_POS_RPE}')
print(f'RESTRICT_TRIALS_TO_POS_RPE: {RESTRICT_TRIALS_TO_POS_RPE}')
print(f'TRIAL_FILTER_LABEL    : {TRIAL_FILTER_LABEL}')
print(f'min_trials_per_cond_effective: {min_trials_per_cond_effective()}')
print(f'PAC_PRIMITIVE         : {PAC_PRIMITIVE}')


In [ ]:
# -----------------------------
# Inline helpers
# -----------------------------


def _window_mask(times: np.ndarray, window: Optional[Tuple[float, float]]) -> np.ndarray:
    if window is None:
        return np.ones_like(times, dtype=bool)
    t0, t1 = float(window[0]), float(window[1])
    return (times >= t0) & (times <= t1)


def load_haufe_patterns_table(csv_path: Path) -> pd.DataFrame:
    """Read the posthoc Haufe-patterns CSV. Returns a DataFrame with original columns."""
    df = pd.read_csv(csv_path)
    required = {'ROI', 'Contrast', 'Subject', 'Electrode', 'PatternValue', 'WeightValue'}
    missing = required - set(df.columns)
    if missing:
        raise ValueError(f'Patterns CSV missing required columns: {missing}')
    df['Subject'] = df['Subject'].astype(str)
    df['Electrode'] = df['Electrode'].astype(str)
    return df


def subject_acc_haufe_vector(
    patterns_df: pd.DataFrame,
    subject_id: str,
    contrast: str,
    electrode_labels: np.ndarray,
    *,
    value_col: str = 'PatternValue',
    min_kept: int = 2,
) -> Tuple[Optional[np.ndarray], Optional[np.ndarray], Optional[np.ndarray]]:
    """Inner-join CSV electrodes with `electrode_labels`.

    Returns `(w_subj, kept_labels, kept_indices)` aligned to the cue-HFA electrode order, or
    `(None, None, None)` if fewer than `min_kept` electrodes match.
    """
    sub = patterns_df[
        (patterns_df['ROI'] == 'ACC')
        & (patterns_df['Contrast'] == contrast)
        & (patterns_df['Subject'] == subject_id)
    ]
    if len(sub) == 0:
        return None, None, None
    pmap: Dict[str, float] = {}
    for lab, val in zip(sub['Electrode'].astype(str), sub[value_col].astype(float)):
        if np.isfinite(val):
            pmap[lab] = float(val)
    elec_array = np.asarray(electrode_labels, dtype=str)
    keep_mask = np.array([lbl in pmap for lbl in elec_array], dtype=bool)
    if int(keep_mask.sum()) < int(min_kept):
        return None, None, None
    kept_indices = np.where(keep_mask)[0].astype(np.int64)
    kept_labels = elec_array[kept_indices]
    w = np.asarray([pmap[lbl] for lbl in kept_labels], dtype=np.float64)
    return w, kept_labels, kept_indices


def compute_acc_reactivation_scalar(
    cue_hfa_3d: np.ndarray, w_subj: np.ndarray,
) -> np.ndarray:
    """Haufe-weighted sum across ACC electrodes -> scalar cue reactivation `(n_trials, n_t)`."""
    cue = np.asarray(cue_hfa_3d, dtype=np.float64)
    w = np.asarray(w_subj, dtype=np.float64)
    if cue.ndim != 3:
        raise ValueError(f'cue_hfa_3d must be 3D, got {cue.shape}')
    if w.ndim != 1 or w.shape[0] != cue.shape[1]:
        raise ValueError(f'w_subj shape {w.shape} incompatible with cue {cue.shape}')
    return np.einsum('tec,e->tc', cue, w)


def compute_acc_reactivation_elec(
    cue_hfa_3d: np.ndarray, w_subj: np.ndarray,
) -> np.ndarray:
    """Per-electrode Haufe-scaled cue HFA: `r[t,e,t] = cue_hfa[t,e,t] * w[e]`."""
    cue = np.asarray(cue_hfa_3d, dtype=np.float64)
    w = np.asarray(w_subj, dtype=np.float64)
    if cue.ndim != 3:
        raise ValueError(f'cue_hfa_3d must be 3D, got {cue.shape}')
    if w.ndim != 1 or w.shape[0] != cue.shape[1]:
        raise ValueError(f'w_subj shape {w.shape} incompatible with cue {cue.shape}')
    return cue * w[None, :, None]


def compute_per_electrode_acc_pac(
    phase_mv: np.ndarray,
    cue_hfa_3d: np.ndarray,
    w_subj: np.ndarray,
    *,
    n_perm: int = 250,
    random_state: int = 0,
    store_surr_distribution: bool = False,
) -> Dict[str, Any]:
    """Per-electrode Tort MI PAC: ACC theta phase vs Haufe-scaled HFA on the same electrode.

    Uses ``Pac(idpac=(2, 1, 0))`` (no Gaussian copula). Per-trial z from surrogate mean/std.

    If ``store_surr_distribution`` is True, ``surr_distribution_elec`` has shape
    ``(n_elec, n_perm, n_trials)`` (float32); otherwise it is None to save memory in batch runs.
    """
    ph = np.asarray(phase_mv, dtype=np.float64)
    cue = np.asarray(cue_hfa_3d, dtype=np.float64)
    w = np.asarray(w_subj, dtype=np.float64)
    if ph.shape != cue.shape:
        raise ValueError(f'shape mismatch phase={ph.shape} cue={cue.shape}')
    react = compute_acc_reactivation_elec(cue, w)
    n_trials, n_elec, _n_t = ph.shape

    trial_pac_z_elec = np.full((n_trials, n_elec), np.nan, dtype=np.float64)
    pac_raw_elec = np.full((n_trials, n_elec), np.nan, dtype=np.float64)
    surr_mean_elec = np.full((n_trials, n_elec), np.nan, dtype=np.float64)
    surr_std_elec = np.full((n_trials, n_elec), np.nan, dtype=np.float64)
    surr_stack = []

    for e in range(n_elec):
        pha_e = ph[:, e, :][None, :, :]
        amp_e = react[:, e, :][None, :, :]
        pac_obj = Pac(
            idpac=(2, 1, 0),
            f_pha=[[4.0, 8.0]],
            f_amp=[[70.0, 200.0]],
            dcomplex='wavelet',
        )
        epoch_pac = pac_obj.fit(
            pha_e, amp_e,
            n_perm=int(n_perm), n_jobs=1,
            random_state=int(random_state) + int(e),
        )
        ep = np.squeeze(np.asarray(epoch_pac, dtype=np.float64), axis=(0, 1))
        surr = pac_obj.surrogates
        if surr is None:
            sm = np.zeros_like(ep)
            ss = np.ones_like(ep)
        else:
            surr_s = np.squeeze(np.asarray(surr, dtype=np.float64), axis=(1, 2))
            sm = surr_s.mean(axis=0)
            ss = surr_s.std(axis=0)
            ss = np.where(ss > 0, ss, np.nan)
            if store_surr_distribution:
                surr_stack.append(surr_s.astype(np.float32))
        with np.errstate(divide='ignore', invalid='ignore'):
            z_e = (ep - sm) / ss
        pac_raw_elec[:, e] = ep
        trial_pac_z_elec[:, e] = z_e
        surr_mean_elec[:, e] = sm
        surr_std_elec[:, e] = ss

    surr_distribution_elec = (
        np.stack(surr_stack, axis=0).astype(np.float32) if surr_stack else None
    )
    trial_pac_z = np.nanmean(trial_pac_z_elec, axis=1)
    pac_raw_mean = np.nanmean(pac_raw_elec, axis=1)
    surr_mean_mean = np.nanmean(surr_mean_elec, axis=1)
    surr_std_mean = np.nanmean(surr_std_elec, axis=1)

    return {
        'trial_pac_z_elec': trial_pac_z_elec,
        'trial_pac_z': trial_pac_z.astype(np.float64),
        'pac_raw_elec': pac_raw_elec,
        'pac_raw': pac_raw_mean.astype(np.float64),
        'surr_mean': surr_mean_mean.astype(np.float64),
        'surr_std': surr_std_mean.astype(np.float64),
        'surr_mean_elec': surr_mean_elec.astype(np.float64),
        'surr_std_elec': surr_std_elec.astype(np.float64),
        'surr_distribution_elec': surr_distribution_elec,
        'n_perm': int(n_perm),
    }


def build_synthetic_pac_dataset(seed: int = 7, n_trials: int = 80,
                                n_acc_elec: int = 4, sfreq: float = 200.0,
                                dur_s: float = 1.5, theta_hz: float = 6.0,
                                preferred_phase: float = 0.5,
                                modulation: float = 1.0,
                                noise: float = 0.2,
                                ) -> Dict[str, np.ndarray]:
    """Construct a synthetic (phase, lfp, locked, unlocked) dataset for the methods
    walkthrough.

    Returns a dict with arrays shaped ``(n_trials, n_acc_elec, n_t)`` for
    ``phase_mv``, ``lfp_mv``, ``locked_react``, ``unlocked_react`` plus a 1D
    ``times`` array (seconds).

    - ``phase_mv``         : multivariate theta phase (radians, in [-pi, pi]).
    - ``lfp_mv``           : synthetic theta-filtered LFP = cos(phase_mv) so panel 1
                             has an analog of the real notch-and-bandpassed LFP.
    - ``locked_react``     : reactivation strongly modulated by phase
                             ``cos(phi - preferred_phase)``.
    - ``unlocked_react``   : reactivation independent of phase (Gaussian noise).
    """
    rng = np.random.default_rng(int(seed))
    n_t = int(sfreq * dur_s)
    times = np.arange(n_t) / float(sfreq)
    elec_lags = rng.uniform(-0.05, 0.05, size=int(n_acc_elec))

    phase_mv = np.zeros((n_trials, n_acc_elec, n_t), dtype=np.float64)
    lfp_mv = np.zeros_like(phase_mv)
    locked = np.zeros_like(phase_mv)
    unlocked = np.zeros_like(phase_mv)
    for tr in range(int(n_trials)):
        tr_lag = rng.uniform(-0.05, 0.05)
        for e in range(int(n_acc_elec)):
            phi = 2 * np.pi * float(theta_hz) * (times + tr_lag + elec_lags[e])
            phi = phi + rng.normal(0, 0.05, n_t)
            phi = (phi + np.pi) % (2 * np.pi) - np.pi
            phase_mv[tr, e] = phi
            lfp_mv[tr, e] = np.cos(phi)
            locked[tr, e] = (
                1.0 + float(modulation) * np.cos(phi - float(preferred_phase))
                + rng.normal(0, float(noise), n_t)
            )
            unlocked[tr, e] = rng.normal(1.0, 1.0, n_t)
    return {
        'phase_mv': phase_mv,
        'lfp_mv': lfp_mv,
        'locked_react': locked,
        'unlocked_react': unlocked,
        'times': times,
    }

# ---- I/O / cache helpers ----

def _npz_is_current(npz_path: Path) -> bool:
    """Return True iff a per-subject NPZ exists with the current PAC_PRIMITIVE marker."""
    if not npz_path.exists():
        return False
    try:
        d = np.load(npz_path, allow_pickle=True)
        need = {
            'PAC_PRIMITIVE', 'SELECTED_CONTRAST', 'reactivation_elec',
            'trial_pac_z_elec', 'trial_pac_z', 'trial_filter',
        }
        if not need.issubset(set(d.files)):
            return False
        if str(d['PAC_PRIMITIVE']) != PAC_PRIMITIVE:
            return False
        if str(d['SELECTED_CONTRAST']) != SELECTED_CONTRAST:
            return False
        tf = d['trial_filter']
        tf_s = str(np.asarray(tf).reshape(-1)[0]) if np.asarray(tf).size else ''
        if tf_s != TRIAL_FILTER_LABEL:
            return False
        return True
    except Exception:
        return False


In [ ]:
# -----------------------------
# Subject list
# -----------------------------
demo_file = f'{base_dir}/projects/guLab/Salman/EMU/MemoryBanditsDatabase.xlsx'
demo_data = pd.read_excel(demo_file, engine='openpyxl')
subj_df = demo_data[(demo_data.LongVersion == 1) & (demo_data.Minerva == 1)].copy()
subj_df = subj_df[~subj_df['MSSMCode'].isin(['MS015', 'MS024', 'MS034', 'MS038'])].copy()
subjects = sorted(subj_df['MSSMCode'].dropna().astype(str).unique().tolist())

haufe_patterns_df = load_haufe_patterns_table(HAUFE_PATTERNS_CSV)
csv_subjects = sorted(set(haufe_patterns_df['Subject'].astype(str)))
subjects_with_patterns = [s for s in subjects if s in set(csv_subjects)]

print('Eligible subjects total       :', len(subjects))
print('Subjects with Haufe patterns  :', len(subjects_with_patterns))
print('Sample subjects with patterns :', subjects_with_patterns[:10])

# Quick contrast/subject summary
ctab = (
    haufe_patterns_df.groupby(['Contrast', 'Subject'])['Electrode']
    .nunique()
    .reset_index()
    .rename(columns={'Electrode': 'n_elec'})
)
print()
print('Contrast x Subject electrode counts (head):')
print(ctab.head(12).to_string(index=False))


In [ ]:
# -----------------------------
# Per-subject runner
# -----------------------------


def _build_subject(subject_id: str) -> Subject:
    return Subject(
        subject=subject_id, base_dir=base_dir, load_dir=load_dir,
        min_trials=MIN_TRIALS, verbose=False,
    )


def _load_acc_cue_hfa(subj: Subject) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
    signal, times, elec_labels = _get_smoothed_event_signal(
        subject_obj=subj, roi=ACC_ROI, event=EVENT_CUE,
        freq_range=HFA_BAND, downsample_factor=DOWNSAMPLE_FACTOR,
        win_ms=WIN_MS, smooth_method=HFA_SMOOTH_METHOD,
        artifact_abs_thresh=HFA_ARTIFACT_ABS_THRESH,
    )
    times = np.asarray(times, dtype=np.float64)
    mask_t = _window_mask(times, CUE_WINDOW)
    sig = np.asarray(signal[..., mask_t], dtype=np.float64)
    return sig, times[mask_t], np.asarray(elec_labels, dtype=str)


def _load_hit_miss_for_cue(subj: Subject) -> pd.DataFrame:
    epochs_file = os.path.join(
        subj.base_dir, 'projects', 'guLab', 'Salman', 'EphysAnalyses',
        subj.subject, 'neural', 'Day1', 'cue_on-epo.fif',
    )
    if not os.path.exists(epochs_file):
        raise FileNotFoundError(epochs_file)
    epochs = mne.read_epochs(epochs_file, preload=False, verbose=False)
    n_epochs = int(len(epochs))
    subj._ensure_behavior_loaded()
    return build_hit_miss_trial_map(
        subj._combined_df, subj_id=subj.subject,
        n_epochs=n_epochs, event='cue_on',
    )


def _pos_rpe_epoch_mask(behavior_df: pd.DataFrame, subj_id: str, n_trials: int,
                        day: str = 'Day1') -> np.ndarray:
    """Boolean mask (length n_trials) for cue_on epochs in the positive RPE tercile (`rpe_cat=='pos'`)."""
    rpe_map = build_rpe_trial_map(
        behavior_df, subj_id=subj_id, n_epochs=int(n_trials),
        day=day, event='cue_on',
    )
    mask = np.zeros(int(n_trials), dtype=bool)
    if len(rpe_map) == 0 or 'epoch_idx' not in rpe_map.columns or 'rpe_cat' not in rpe_map.columns:
        return mask
    for row in rpe_map.itertuples(index=False):
        if not bool(getattr(row, 'alignment_valid', False)):
            continue
        ei = int(getattr(row, 'epoch_idx'))
        if not (0 <= ei < int(n_trials)):
            continue
        rc = str(getattr(row, 'rpe_cat')).strip().lower()
        if rc == 'pos':
            mask[ei] = True
    return mask


def _load_acc_theta_phase(
    subj: Subject, cue_times: np.ndarray,
) -> Tuple[np.ndarray, np.ndarray]:
    """Return (phase_per_trial_elec_t, electrode_labels).

    Wraps `extract_theta_phase_from_cue_epochs` and transposes the (n_elec, n_t, n_trials)
    output to (n_trials, n_elec, n_t).
    """
    ph_out = extract_theta_phase_from_cue_epochs(
        subject_obj=subj, roi=ACC_ROI, event=EVENT_CUE,
        freq_band=THETA_BAND,
        target_times=np.asarray(cue_times, dtype=np.float32),
    )
    phi_etc = np.asarray(ph_out['phase'], dtype=np.float64)          # (n_elec, n_t, n_trials)
    phi_per_trial = np.transpose(phi_etc, (2, 0, 1))                 # (n_trials, n_elec, n_t)
    elec_labels = np.asarray(ph_out['electrode_labels'], dtype=str)
    return phi_per_trial, elec_labels


def run_subject_acc_pac(subject_id: str, overwrite: bool = False) -> Dict[str, Any]:
    out_npz = per_subject_output_root / f'{subject_id}_acc_thetapac_reactivation.npz'
    if (not overwrite) and _npz_is_current(out_npz):
        return {'subject': subject_id, 'status': 'cached', 'npz_path': str(out_npz)}

    try:
        subj = _build_subject(subject_id)
    except Exception as exc:
        return {'subject': subject_id, 'status': 'fail', 'reason': f'subject init: {exc!r}'}

    # ACC cue HFA (n_trials, n_acc_elec, n_t)
    try:
        acc_cue_hfa, cue_times, acc_hfa_labels = _load_acc_cue_hfa(subj)
    except Exception as exc:
        return {'subject': subject_id, 'status': 'fail', 'reason': f'load ACC HFA: {exc!r}'}

    # Subject Haufe vector aligned to HFA electrodes
    w_subj, kept_labels, kept_idx = subject_acc_haufe_vector(
        haufe_patterns_df, subject_id, SELECTED_CONTRAST, acc_hfa_labels,
        value_col=HAUFE_WEIGHT_COLUMN, min_kept=MIN_PATTERN_ELECTRODES,
    )
    if w_subj is None:
        return {'subject': subject_id, 'status': 'skip',
                'reason': f'<{MIN_PATTERN_ELECTRODES} matched Haufe electrodes'}

    acc_cue_hfa = acc_cue_hfa[:, kept_idx, :]
    n_trials, n_elec_acc, n_t = acc_cue_hfa.shape

    # Hit / miss map (full cue epoch count before optional RPE restriction)
    try:
        hit_map = _load_hit_miss_for_cue(subj)
        is_hit = (hit_map['hit_miss'].astype(str) == 'hit').to_numpy()
        is_miss = (hit_map['hit_miss'].astype(str) == 'miss').to_numpy()
        n_epochs_beh = len(hit_map)
    except Exception as exc:
        return {'subject': subject_id, 'status': 'fail', 'reason': f'hit/miss map: {exc!r}'}
    if n_epochs_beh != n_trials:
        return {'subject': subject_id, 'status': 'fail',
                'reason': f'epoch count mismatch behavior={n_epochs_beh} hfa={n_trials}'}

    n_trials_pre_rpe = int(n_trials)
    pos_rpe_mask_orig: Optional[np.ndarray] = None
    if RESTRICT_TRIALS_TO_POS_RPE:
        pos_rpe_mask_orig = _pos_rpe_epoch_mask(subj._combined_df, subj.subject, n_trials)
        n_pos = int(np.count_nonzero(pos_rpe_mask_orig))
        if n_pos == 0:
            return {'subject': subject_id, 'status': 'skip',
                    'reason': 'no positive-RPE (tercile pos) trials after alignment'}
        acc_cue_hfa = acc_cue_hfa[pos_rpe_mask_orig]
        is_hit = is_hit[pos_rpe_mask_orig]
        is_miss = is_miss[pos_rpe_mask_orig]
        n_trials = int(acc_cue_hfa.shape[0])

    min_k = int(min_trials_per_cond_effective())
    cond_indices = {
        'hit': np.where(is_hit)[0].astype(np.int64),
        'miss': np.where(is_miss)[0].astype(np.int64),
    }
    if cond_indices['hit'].size < min_k or cond_indices['miss'].size < min_k:
        return {'subject': subject_id, 'status': 'skip',
                'reason': (
                    f'n_hit={cond_indices["hit"].size}, n_miss={cond_indices["miss"].size} '
                    f'< min_trials_per_cond_effective()={min_k} '
                    f'(pos_RPE={RESTRICT_TRIALS_TO_POS_RPE})'
                )}

    # Multivariate ACC theta phase on the cue-time grid (must align to HFA times)
    try:
        phase_mv, acc_phase_labels = _load_acc_theta_phase(subj, cue_times)
    except Exception as exc:
        return {'subject': subject_id, 'status': 'fail', 'reason': f'load ACC theta phase: {exc!r}'}

    if RESTRICT_TRIALS_TO_POS_RPE:
        if pos_rpe_mask_orig is None:
            return {'subject': subject_id, 'status': 'fail', 'reason': 'internal: pos_rpe_mask_orig unset'}
        if phase_mv.shape[0] != int(pos_rpe_mask_orig.shape[0]):
            return {'subject': subject_id, 'status': 'fail',
                    'reason': (
                        f'phase/HFA trial mismatch for RPE mask: phase_trials={phase_mv.shape[0]} '
                        f"mask_len={int(pos_rpe_mask_orig.shape[0])}"
                    )}
        phase_mv = phase_mv[pos_rpe_mask_orig]

    # Intersect phase electrodes with HFA/Haufe electrodes; per-electrode reactivation uses same w.
    common_labels, hfa_pos, pha_pos = np.intersect1d(kept_labels, acc_phase_labels,
                                                     return_indices=True)
    if common_labels.size < MIN_PATTERN_ELECTRODES:
        return {'subject': subject_id, 'status': 'skip',
                'reason': f'<{MIN_PATTERN_ELECTRODES} ACC electrodes common to HFA + phase'}
    acc_cue_hfa = acc_cue_hfa[:, hfa_pos, :]
    phase_mv = phase_mv[:, pha_pos, :]
    kept_labels = common_labels.astype(str)
    w_subj = w_subj[hfa_pos]

    reactivation_elec = compute_acc_reactivation_elec(acc_cue_hfa, w_subj)

    if not (np.isfinite(reactivation_elec).all() and np.isfinite(phase_mv).all()):
        trial_ok = (
            np.all(np.isfinite(reactivation_elec).reshape(reactivation_elec.shape[0], -1), axis=1)
            & np.all(np.isfinite(phase_mv).reshape(phase_mv.shape[0], -1), axis=1)
        )
        if int(trial_ok.sum()) < (cond_indices['hit'].size + cond_indices['miss'].size) * 0.5:
            return {'subject': subject_id, 'status': 'fail',
                    'reason': f'too many NaN trials ({int((~trial_ok).sum())})'}
        keep_idx = np.where(trial_ok)[0]
        reactivation_elec = reactivation_elec[keep_idx]
        acc_cue_hfa = acc_cue_hfa[keep_idx]
        phase_mv = phase_mv[keep_idx]
        old_to_new = -np.ones(n_trials, dtype=np.int64)
        old_to_new[keep_idx] = np.arange(keep_idx.size, dtype=np.int64)
        for cond in cond_indices:
            new = old_to_new[cond_indices[cond]]
            cond_indices[cond] = new[new >= 0]
        n_trials = reactivation_elec.shape[0]
        min_k2 = int(min_trials_per_cond_effective())
        if cond_indices['hit'].size < min_k2 or cond_indices['miss'].size < min_k2:
            return {'subject': subject_id, 'status': 'skip',
                    'reason': (
                        f'after NaN drop: n_hit={cond_indices["hit"].size}, '
                        f'n_miss={cond_indices["miss"].size} < min_k={min_k2}'
                    )}

    pac_out = compute_per_electrode_acc_pac(
        phase_mv=phase_mv,
        cue_hfa_3d=acc_cue_hfa,
        w_subj=w_subj,
        n_perm=N_PAC_PERM,
        random_state=PAC_RANDOM_STATE,
        store_surr_distribution=False,
    )

    payload = {
        'PAC_PRIMITIVE': PAC_PRIMITIVE,
        'SELECTED_CONTRAST': SELECTED_CONTRAST,
        'trial_filter': np.array(TRIAL_FILTER_LABEL, dtype=object),
        'n_trials_pre_rpe_filter': np.int64(n_trials_pre_rpe),
        'subject': subject_id,
        'cue_times': cue_times.astype(np.float64),
        'acc_electrode_labels': kept_labels.astype(object),
        'w_subj': w_subj.astype(np.float64),
        'reactivation_elec': reactivation_elec.astype(np.float32),
        'trial_pac_z_elec': pac_out['trial_pac_z_elec'].astype(np.float64),
        'trial_pac_z': pac_out['trial_pac_z'].astype(np.float64),
        'pac_raw': pac_out['pac_raw'].astype(np.float64),
        'pac_raw_elec': pac_out['pac_raw_elec'].astype(np.float64),
        'surr_mean': pac_out['surr_mean'].astype(np.float64),
        'surr_std': pac_out['surr_std'].astype(np.float64),
        'n_perm': np.int64(pac_out['n_perm']),
        'hit_indices': cond_indices['hit'].astype(np.int64),
        'miss_indices': cond_indices['miss'].astype(np.int64),
        'n_trials': np.int64(n_trials),
        'n_hit': np.int64(cond_indices['hit'].size),
        'n_miss': np.int64(cond_indices['miss'].size),
    }
    np.savez(out_npz, **payload)
    return {
        'subject': subject_id, 'status': 'ok', 'npz_path': str(out_npz),
        'trial_pac_z': pac_out['trial_pac_z'],
        'trial_pac_z_elec': pac_out['trial_pac_z_elec'],
        'pac_raw': pac_out['pac_raw'],
        'cue_times': cue_times,
        'acc_electrode_labels': kept_labels,
        'w_subj': w_subj,
        'reactivation_elec': reactivation_elec,
        'cond_indices': cond_indices,
        'n_trials': int(n_trials),
        'n_hit': int(cond_indices['hit'].size),
        'n_miss': int(cond_indices['miss'].size),
    }


def load_subject_npz(subject_id: str) -> Optional[Dict[str, Any]]:
    out_npz = per_subject_output_root / f'{subject_id}_acc_thetapac_reactivation.npz'
    if not _npz_is_current(out_npz):
        return None
    return dict(np.load(out_npz, allow_pickle=True))


## Methods walkthrough

Two coordinated passes that let a naive reader inspect the analysis at every level:

- **Pass 1 - synthetic walkthrough.** Build a small synthetic dataset with known
  theta phase and two amplitude conditions: **locked** per-electrode reactivation
  coupled to each electrode's own phase (`cos(phi_e - preferred)`) and **unlocked**
  Gaussian noise per electrode. Six panels visualize (1) a single-electrode time series,
  (2) multivariate phase and per-electrode locked heatmaps for one trial (viz),
  (3) same-electrode phase vs Haufe-scaled locked amplitude for that trial,
  (4) polar bins on that pair, (5) swap-phase surrogate null vs observed **Tort MI**
  PAC for one electrode/trial, and (6) the per-trial aggregate PAC z (mean over
  electrodes) for locked vs unlocked. The `mean_z_locked > mean_z_unlocked`
  assertion stays in this cell.

- **Pass 2 - real subject mirror (MS012).** Same six panels on real data: per-electrode
  Tort MI PAC (`Pac(idpac=(2, 1, 0))`) with ACC theta phase and **Haufe-scaled** cue HFA
  on each electrode; see subsection below.

Both passes save a figure to
`analysis_outputs/acc_thetapac_reactivation/<SELECTED_CONTRAST>/figures/`:
`walkthrough_synthetic.png` and `walkthrough_real_<subject>.png`.

### Pass 1 - synthetic walkthrough with ground truth


In [ ]:
# -----------------------------
# Pass 1: synthetic methods walkthrough (per-electrode Tort MI PAC)
# -----------------------------

syn = build_synthetic_pac_dataset(seed=7, n_trials=80, n_acc_elec=4,
                                  sfreq=200.0, dur_s=1.5, theta_hz=6.0,
                                  preferred_phase=0.5, modulation=1.0, noise=0.2)
phase_syn = syn['phase_mv']
lfp_syn = syn['lfp_mv']
locked_react = syn['locked_react']
unlocked_react = syn['unlocked_react']
times_syn = syn['times']

w_demo = np.asarray([1.0, -0.5, 0.25, -0.25], dtype=np.float64)
w_demo = w_demo / np.sqrt(np.sum(w_demo ** 2))

N_PERM_DEMO = 200
res_locked = compute_per_electrode_acc_pac(
    phase_mv=phase_syn, cue_hfa_3d=locked_react, w_subj=w_demo,
    n_perm=N_PERM_DEMO, random_state=7, store_surr_distribution=True,
)
res_unlocked = compute_per_electrode_acc_pac(
    phase_mv=phase_syn, cue_hfa_3d=unlocked_react, w_subj=w_demo,
    n_perm=N_PERM_DEMO, random_state=7, store_surr_distribution=True,
)

print('Synthetic: phase', phase_syn.shape, '| locked_react', locked_react.shape)
print(f'  trial_pac_z (locked)   mean: {np.nanmean(res_locked["trial_pac_z"]):+.3f}')
print(f'  trial_pac_z (unlocked) mean: {np.nanmean(res_unlocked["trial_pac_z"]):+.3f}')

z_locked = res_locked['trial_pac_z']
trial_idx_syn = int(np.argmax(np.abs(z_locked)))
elec_idx_syn = 0
locked_haufe = locked_react * w_demo[None, :, None]
phi_e = phase_syn[trial_idx_syn, elec_idx_syn]
r_e = locked_haufe[trial_idx_syn, elec_idx_syn]

fig = plt.figure(figsize=(13, 11))
gs = fig.add_gridspec(3, 3, height_ratios=[1.0, 1.0, 1.0],
                      width_ratios=[1.0, 1.1, 1.0])

# Panel 1
ax1 = fig.add_subplot(gs[0, 0])
ax1.plot(times_syn, lfp_syn[trial_idx_syn, elec_idx_syn], lw=1.0,
         color='steelblue', label='theta LFP')
ax1.plot(times_syn, locked_react[trial_idx_syn, elec_idx_syn], lw=1.3,
         color='firebrick', label='per-elec locked (viz)')
ax1.set_xlabel('time (s)')
ax1.set_ylabel('amplitude (a.u.)')
ax1.set_title(
    f'Panel 1 - trial {trial_idx_syn}, elec {elec_idx_syn} '
    f'(PAC: phase vs w_demo·amp on each electrode)'
)
ax2_twin = ax1.twinx()
ax2_twin.plot(times_syn, phi_e,
              color='gray', alpha=0.6, lw=0.7, label='theta phase')
ax2_twin.set_ylim(-np.pi, np.pi)
ax2_twin.set_ylabel('theta phase (rad)')
ax1.legend(loc='upper left', fontsize=7, frameon=False)
ax2_twin.legend(loc='upper right', fontsize=7, frameon=False)
sns.despine(ax=ax1, right=False)

# Panel 2a/b
ax2a = fig.add_subplot(gs[0, 1])
ax2b = fig.add_subplot(gs[1, 1])
extent = [times_syn[0], times_syn[-1], phase_syn.shape[1] - 0.5, -0.5]
im_ph = ax2a.imshow(phase_syn[trial_idx_syn], aspect='auto', cmap='twilight',
                    vmin=-np.pi, vmax=np.pi, extent=extent)
ax2a.set_title(f'Panel 2a - theta phase (trial {trial_idx_syn})')
ax2a.set_ylabel('electrode')
ax2a.set_xlabel('time (s)')
plt.colorbar(im_ph, ax=ax2a, fraction=0.04, pad=0.02, label='phase (rad)')
im_r = ax2b.imshow(locked_haufe[trial_idx_syn], aspect='auto', cmap='magma', extent=extent)
ax2b.set_title('Panel 2b - Haufe-scaled locked amp (w_demo·locked)')
ax2b.set_ylabel('electrode')
ax2b.set_xlabel('time (s)')
plt.colorbar(im_r, ax=ax2b, fraction=0.04, pad=0.02, label='amp (a.u.)')

# Panel 3: same-electrode phase vs Haufe-scaled amplitude (same trial)
ax3 = fig.add_subplot(gs[1, 0])
sc = ax3.scatter(phi_e, r_e, c=times_syn, s=6, cmap='viridis', alpha=0.6)
ax3.set_xlim(-np.pi, np.pi)
ax3.set_xlabel(f'theta phase elec {elec_idx_syn} (rad)')
ax3.set_ylabel('Haufe-scaled locked amp (same electrode)')
plt.colorbar(sc, ax=ax3, fraction=0.04, pad=0.02, label='time (s)')
sns.despine(ax=ax3)

# Panel 4 - polar bins on same pair
n_bins = 18
edges = np.linspace(-np.pi, np.pi, n_bins + 1)
centers = 0.5 * (edges[:-1] + edges[1:])
phi_flat = phi_e.ravel()
r_flat = r_e.ravel()
bin_idx = np.clip(np.digitize(phi_flat, edges) - 1, 0, n_bins - 1)
bin_mean = np.array([
    float(np.nanmean(r_flat[bin_idx == b])) if int((bin_idx == b).sum()) > 0 else np.nan
    for b in range(n_bins)
])
ax4 = fig.add_subplot(gs[0, 2], projection='polar')
ax4.bar(centers, bin_mean - np.nanmin(bin_mean),
        width=np.diff(edges), align='center', alpha=0.7,
        color='firebrick', edgecolor='k', linewidth=0.4)
ax4.set_title('Panel 4 - polar (same elec)', y=1.08)
ax4.set_yticklabels([])

# Panel 5
ax5 = fig.add_subplot(gs[2, 0])
surr_loc = res_locked['surr_distribution_elec']
if surr_loc is not None:
    surr_trial = surr_loc[elec_idx_syn, :, trial_idx_syn]
    ax5.hist(surr_trial, bins=30, color='gray', alpha=0.7,
             edgecolor='black', linewidth=0.4, label='swap-phase surrogates')
    obs_val = res_locked['pac_raw_elec'][trial_idx_syn, elec_idx_syn]
    ax5.axvline(obs_val, color='firebrick', lw=2.0, label='observed PAC')
    z_e = res_locked['trial_pac_z_elec'][trial_idx_syn, elec_idx_syn]
    ax5.set_title(
        f'Panel 5 - null vs observed (trial {trial_idx_syn}, elec {elec_idx_syn})\n'
        f'z_e = {z_e:+.2f} | z_agg = {res_locked["trial_pac_z"][trial_idx_syn]:+.2f}'
    )
    ax5.set_xlabel('raw Tort MI PAC')
    ax5.set_ylabel('surrogate count')
    ax5.legend(fontsize=7, frameon=False)
sns.despine(ax=ax5)

# Panel 6
ax6 = fig.add_subplot(gs[2, 1:])
sns.kdeplot(z_locked, ax=ax6, label='locked (agg z)', fill=True, alpha=0.45, color='firebrick')
sns.kdeplot(res_unlocked['trial_pac_z'], ax=ax6, label='unlocked (agg z)', fill=True,
            alpha=0.45, color='steelblue')
ax6.axvline(0.0, color='k', lw=0.5, ls='--')
ax6.axvline(float(np.nanmean(z_locked)), color='firebrick', lw=1.0, ls=':')
ax6.axvline(float(np.nanmean(res_unlocked['trial_pac_z'])),
            color='steelblue', lw=1.0, ls=':')
ax6.set_xlabel('per-trial PAC z (mean over electrodes)')
ax6.set_title(
    f'Panel 6 - per-trial z (mean locked={np.nanmean(z_locked):+.3f}, '
    f'unlocked={np.nanmean(res_unlocked["trial_pac_z"]):+.3f})'
)
ax6.legend(frameon=False, fontsize=8)
sns.despine(ax=ax6)

fig.suptitle(
    'Pass 1: synthetic  -  per-electrode Tort MI PAC (locked vs noise)',
    fontsize=12, y=1.0,
)
fig.tight_layout()
syn_fig_path = figure_output_root / 'walkthrough_synthetic.png'
fig.savefig(syn_fig_path, dpi=160, bbox_inches='tight')
print('saved', syn_fig_path)
plt.show()

mean_z_locked = float(np.nanmean(z_locked))
mean_z_unlocked = float(np.nanmean(res_unlocked['trial_pac_z']))
print(f'mean trial PAC z (locked)   : {mean_z_locked:+.3f}')
print(f'mean trial PAC z (unlocked) : {mean_z_unlocked:+.3f}')
assert mean_z_locked > mean_z_unlocked, (
    'Synthetic recovery failed: expected mean trial PAC z (locked) > (unlocked)'
)
print('synthetic recovery OK')


## Group loop

Run every eligible subject in parallel. Subjects with a current cached NPZ are reused; the
schema marker (`PAC_PRIMITIVE`), `SELECTED_CONTRAST`, and `trial_filter` guard against stale caches.

In [ ]:
def _safe_run(subject_id: str) -> Dict[str, Any]:
    try:
        return run_subject_acc_pac(subject_id, overwrite=False)
    except Exception as exc:
        return {'subject': subject_id, 'status': 'fail', 'reason': f'top-level: {exc!r}'}


group_results: List[Dict[str, Any]] = Parallel(
    n_jobs=N_JOBS, backend=JOBLIB_BACKEND, verbose=5,
)(delayed(_safe_run)(s) for s in subjects_with_patterns)

ok_subjects = [r['subject'] for r in group_results if r.get('status') in {'ok', 'cached'}]
fail_subjects = [(r['subject'], r.get('reason')) for r in group_results if r.get('status') == 'fail']
skip_subjects = [(r['subject'], r.get('reason')) for r in group_results if r.get('status') == 'skip']

print(f'OK subjects ({len(ok_subjects)}):')
for s in ok_subjects:
    print(f'  - {s}')
print(f'\nSkipped subjects ({len(skip_subjects)}):')
for s, why in skip_subjects:
    print(f'  - {s}: {why}')
print(f'\nFailed subjects ({len(fail_subjects)}):')
for s, why in fail_subjects:
    print(f'  - {s}: {why}')


## Group statistics -- hit vs miss

When `RESTRICT_TRIALS_TO_POS_RPE` is True, every subject’s PAC and hit/miss labels are computed on
**positive-RPE cue trials only** (`rpe_cat == 'pos'`, within-subject terciles). That matches the
positive-RPE Haufe contrast but **shrinks trial counts**, so within-subject hit/miss cells can be
small; the notebook still pools subjects for paired tests, but treat those tests as **group-level**
summaries and inspect per-subject `n_hit` / `n_miss` in the CSV.

For each subject we aggregate `trial_pac_z` (mean PAC z across ACC electrodes) by hit / miss into a subject-level summary
DataFrame and run:

- one-sample t-test of each condition's mean z vs 0,
- paired t-test (hit - miss) across subjects,
- Wilcoxon signed-rank as a non-parametric backstop (may fail if too many ties / small n).

The pooled per-trial distribution (across subjects and conditions) is also exported for
descriptive plots.


In [ ]:
rows_subject = []
rows_trial = []
for sid in ok_subjects:
    d = load_subject_npz(sid)
    if d is None:
        continue
    tpz = np.asarray(d['trial_pac_z'], dtype=np.float64)
    hits = np.asarray(d['hit_indices'], dtype=np.int64)
    miss = np.asarray(d['miss_indices'], dtype=np.int64)
    if hits.size == 0 or miss.size == 0:
        continue

    rows_subject.append({
        'subject': sid,
        'n_trials': int(d['n_trials']),
        'n_hit': int(d['n_hit']),
        'n_miss': int(d['n_miss']),
        'mean_z_all': float(np.nanmean(tpz)),
        'mean_z_hit': float(np.nanmean(tpz[hits])),
        'mean_z_miss': float(np.nanmean(tpz[miss])),
        'median_z_hit': float(np.nanmedian(tpz[hits])),
        'median_z_miss': float(np.nanmedian(tpz[miss])),
    })
    for ti in range(tpz.size):
        if ti in set(hits.tolist()):
            cond = 'hit'
        elif ti in set(miss.tolist()):
            cond = 'miss'
        else:
            continue
        rows_trial.append({'subject': sid, 'trial': int(ti),
                           'condition': cond, 'pac_z': float(tpz[ti])})

subject_df = pd.DataFrame(rows_subject)
trial_df = pd.DataFrame(rows_trial)
print(
    f"\nInclusion: {len(subject_df)} subjects with both hit and miss PAC on retained trials "
    f"(RESTRICT_TRIALS_TO_POS_RPE={RESTRICT_TRIALS_TO_POS_RPE})."
)
if RESTRICT_TRIALS_TO_POS_RPE and len(subject_df) > 0:
    print('  n_hit / n_miss in the table are counts on **positive-RPE (tercile) trials** only.')
print('Per-subject summary (head):')
print(subject_df.head(15).to_string(index=False))

stats_rows: List[Dict[str, Any]] = []
if len(subject_df) >= 3:
    for cond in ('hit', 'miss'):
        x = subject_df[f'mean_z_{cond}'].dropna().to_numpy()
        if x.size >= 3:
            t, p = ttest_1samp(x, popmean=0.0)
            stats_rows.append({
                'test': f'one_samp_{cond}_vs_0', 'n': int(x.size),
                'stat': float(t), 'p': float(p),
                'mean': float(np.mean(x)), 'sd': float(np.std(x, ddof=1)),
            })
    paired = subject_df.dropna(subset=['mean_z_hit', 'mean_z_miss']).copy()
    if len(paired) >= 3:
        diff = paired['mean_z_hit'].to_numpy() - paired['mean_z_miss'].to_numpy()
        t, p = ttest_rel(paired['mean_z_hit'].to_numpy(),
                         paired['mean_z_miss'].to_numpy())
        stats_rows.append({
            'test': 'paired_t_hit_vs_miss', 'n': int(len(paired)),
            'stat': float(t), 'p': float(p),
            'mean': float(np.mean(diff)), 'sd': float(np.std(diff, ddof=1)),
        })
        try:
            w_stat, w_p = wilcoxon(paired['mean_z_hit'].to_numpy(),
                                   paired['mean_z_miss'].to_numpy())
            stats_rows.append({
                'test': 'wilcoxon_hit_vs_miss', 'n': int(len(paired)),
                'stat': float(w_stat), 'p': float(w_p),
                'mean': float(np.mean(diff)), 'sd': float(np.std(diff, ddof=1)),
            })
        except Exception as exc:
            print('Wilcoxon failed:', repr(exc))

stats_df = pd.DataFrame(stats_rows)
print('\nGroup statistics:')
print(stats_df.to_string(index=False) if len(stats_df) else '  (insufficient n)')

# Persist outputs
subject_csv = analysis_output_root / 'per_subject_summary.csv'
trial_csv = analysis_output_root / 'per_trial_summary.csv'
stats_csv = analysis_output_root / 'group_stats.csv'
subject_df.to_csv(subject_csv, index=False)
trial_df.to_csv(trial_csv, index=False)
stats_df.to_csv(stats_csv, index=False)
print('\nWrote:')
print(f'  {subject_csv}')
print(f'  {trial_csv}')
print(f'  {stats_csv}')


## Cross-contrast aggregation

Concatenate every `per_subject_summary.csv` found under
`analysis_outputs/acc_thetapac_reactivation/*/`
into one table with a `contrast` column, and save
`all_contrasts_per_subject_summary.csv` at the **parent** of the per-contrast folders.

If both `positive_vs_neutral` and `positive_vs_negative` have been run, also plot a
paired comparison of `mean_z_hit` (or `mean_z_all`) across contrasts for subjects
present in both.


In [ ]:
# -----------------------------
# Cross-contrast aggregation
# -----------------------------
contrast_root = analysis_output_root.parent
assert contrast_root.name == 'acc_thetapac_reactivation', contrast_root

rows_agg = []
for summary_path in sorted(contrast_root.glob('*/per_subject_summary.csv')):
    ctr = summary_path.parent.name
    try:
        df_c = pd.read_csv(summary_path)
    except Exception as exc:
        print('skip', summary_path, repr(exc))
        continue
    df_c = df_c.assign(contrast=ctr)
    rows_agg.append(df_c)

if len(rows_agg) == 0:
    print('No per_subject_summary.csv files found under', contrast_root)
else:
    all_contrast_df = pd.concat(rows_agg, ignore_index=True)
    out_all = contrast_root / 'all_contrasts_per_subject_summary.csv'
    all_contrast_df.to_csv(out_all, index=False)
    print('Wrote', out_all, '| rows:', len(all_contrast_df))

    # Paired slopegraph: mean_z_hit for subjects in exactly two contrasts (neutral + negative)
    piv = all_contrast_df.pivot_table(
        index='subject', columns='contrast', values='mean_z_hit', aggfunc='first',
    )
    want = {'positive_vs_neutral', 'positive_vs_negative'}
    if want.issubset(set(piv.columns)):
        pair_df = piv[list(want)].dropna(how='any')
        if len(pair_df) >= 3:
            fig, ax = plt.subplots(figsize=(4.0, 5.0))
            x0, x1 = 0.0, 1.0
            for subj, row in pair_df.iterrows():
                ax.plot([x0, x1], [row['positive_vs_neutral'], row['positive_vs_negative']],
                        color='lightgray', lw=0.8, zorder=0)
            ax.scatter(np.full(len(pair_df), x0), pair_df['positive_vs_neutral'],
                       s=28, zorder=2, label='pos vs neutral')
            ax.scatter(np.full(len(pair_df), x1), pair_df['positive_vs_negative'],
                       s=28, zorder=2, label='pos vs negative')
            ax.set_xticks([x0, x1])
            ax.set_xticklabels(['pos vs neutral', 'pos vs negative'], rotation=15, ha='right')
            ax.axhline(0.0, color='k', lw=0.5, ls='--', alpha=0.5)
            ax.set_ylabel('mean trial PAC z (hit trials)')
            ax.set_title(f'Paired subjects n={len(pair_df)}')
            ax.legend(frameon=False, fontsize=7)
            sns.despine(ax=ax)
            fig.tight_layout()
            fig.savefig(contrast_root / 'fig_cross_contrast_mean_z_hit.png',
                        dpi=160, bbox_inches='tight')
            print('saved', contrast_root / 'fig_cross_contrast_mean_z_hit.png')
            plt.show()


## Figures

- **Paired strip plot** of subject-level mean PAC z, hit vs miss, with grey lines pairing subjects.
- **Per-trial pooled distribution** (boxplot + strip) of trial-level PAC z by condition.
- **Subject heatmap** of mean PAC z (hit, miss, hit - miss).
- **Trial definition**: figures use per-subject NPZs; when `RESTRICT_TRIALS_TO_POS_RPE` is True,
  `trial_filter == 'pos_rpe_tercile'` and binned-phase / PAC summaries are on that subset.
- **Binned-phase view** of the mean **Haufe-scaled** reactivation (averaged across ACC electrodes
  at each time sample) as a function of *aggregated* ACC theta phase (circular mean across
  electrodes, equal weights), for interpretability.


In [ ]:
def _save(fig, name: str):
    out = figure_output_root / name
    fig.savefig(out, dpi=200, bbox_inches='tight')
    print('saved', out)


# ---- 1. Paired strip plot ----
if len(subject_df) >= 3:
    fig, ax = plt.subplots(figsize=(4.0, 4.5))
    paired = subject_df.dropna(subset=['mean_z_hit', 'mean_z_miss']).copy()
    xs = {'hit': 0.0, 'miss': 1.0}
    rng_jit = np.random.default_rng(0)
    for cond in ('hit', 'miss'):
        x = np.full(len(paired), xs[cond]) + rng_jit.uniform(-0.06, 0.06, len(paired))
        ax.scatter(x, paired[f'mean_z_{cond}'], s=22, alpha=0.85)
    for _, row in paired.iterrows():
        ax.plot([xs['hit'], xs['miss']], [row['mean_z_hit'], row['mean_z_miss']],
                color='lightgray', linewidth=0.8, zorder=0)
    ax.set_xticks([0, 1]); ax.set_xticklabels(['hit', 'miss'])
    ax.axhline(0.0, color='k', lw=0.5, ls='--', alpha=0.6)
    ax.set_ylabel('mean trial PAC z')
    if 'paired_t_hit_vs_miss' in stats_df['test'].values:
        row = stats_df[stats_df.test == 'paired_t_hit_vs_miss'].iloc[0]
        ax.set_title(f"paired t={row['stat']:+.2f}, p={row['p']:.3g} (n={int(row['n'])})")
    sns.despine(ax=ax)
    fig.tight_layout()
    _save(fig, 'fig01_paired_subject_meanz.png')
    plt.show()

# ---- 2. Per-trial pooled distribution ----
if len(trial_df) >= 10:
    fig, ax = plt.subplots(figsize=(4.5, 4.5))
    sns.boxplot(data=trial_df, x='condition', y='pac_z', ax=ax,
                showfliers=False, width=0.5,
                order=['hit', 'miss'])
    sns.stripplot(data=trial_df, x='condition', y='pac_z', ax=ax,
                  order=['hit', 'miss'], color='k', size=1.5, alpha=0.25, jitter=0.2)
    ax.axhline(0.0, color='k', lw=0.5, ls='--', alpha=0.6)
    ax.set_ylabel('per-trial PAC z')
    sns.despine(ax=ax)
    fig.tight_layout()
    _save(fig, 'fig02_pertrial_distribution.png')
    plt.show()

# ---- 3. Subject heatmap of mean z ----
if len(subject_df) >= 3:
    plot_df = subject_df.set_index('subject')[['mean_z_hit', 'mean_z_miss']].copy()
    plot_df['hit_minus_miss'] = plot_df['mean_z_hit'] - plot_df['mean_z_miss']
    plot_df = plot_df.sort_values('hit_minus_miss')
    fig, ax = plt.subplots(figsize=(5.5, max(3.0, 0.3 * len(plot_df))))
    sns.heatmap(plot_df, cmap='vlag', center=0, annot=True, fmt='+.2f', ax=ax)
    ax.set_xlabel('')
    fig.tight_layout()
    _save(fig, 'fig03_subject_heatmap.png')
    plt.show()


# ---- 4. Binned-phase reactivation aggregate ----
def _per_subject_binned_phase(subject_id: str, n_bins: int = 18) -> Optional[Dict[str, np.ndarray]]:
    out_npz = per_subject_output_root / f'{subject_id}_acc_thetapac_reactivation.npz'
    if not _npz_is_current(out_npz):
        return None
    d = np.load(out_npz, allow_pickle=True)
    if 'reactivation_elec' not in d.files:
        return None
    reactivation_elec = np.asarray(d['reactivation_elec'], dtype=np.float64)
    cue_times_npz = np.asarray(d['cue_times'], dtype=np.float64)
    lab_npz = np.asarray(d['acc_electrode_labels'], dtype=str).tolist()

    try:
        subj = _build_subject(subject_id)
        phase_mv, acc_phase_labels = _load_acc_theta_phase(subj, cue_times_npz)
    except Exception:
        return None
    acc_phase_labels = np.asarray(acc_phase_labels, dtype=str)
    try:
        order = np.array([list(acc_phase_labels).index(str(l)) for l in lab_npz], dtype=np.int64)
    except ValueError:
        return None
    phase_mv = phase_mv[:, order, :]

    weights = np.ones(phase_mv.shape[1], dtype=np.float64) / float(phase_mv.shape[1])
    sine = np.sin(phase_mv) * weights[None, :, None]
    cosi = np.cos(phase_mv) * weights[None, :, None]
    agg_phase = np.arctan2(sine.sum(axis=1), cosi.sum(axis=1))

    # Mean Haufe-scaled reactivation across electrodes at each (trial, time).
    mean_react = np.nanmean(reactivation_elec, axis=1)

    edges = np.linspace(-np.pi, np.pi, n_bins + 1)
    bin_idx = np.clip(np.digitize(agg_phase.ravel(), edges) - 1, 0, n_bins - 1)
    react_flat = mean_react.ravel()
    out_mean = np.full(n_bins, np.nan)
    out_std = np.full(n_bins, np.nan)
    out_n = np.zeros(n_bins, dtype=np.int64)
    for b in range(n_bins):
        sel = (bin_idx == b)
        if sel.sum() > 0:
            out_mean[b] = float(np.nanmean(react_flat[sel]))
            out_std[b] = float(np.nanstd(react_flat[sel]))
            out_n[b] = int(sel.sum())
    return {
        'subject': subject_id,
        'bin_centers': 0.5 * (edges[:-1] + edges[1:]),
        'mean': out_mean,
        'std': out_std,
        'n': out_n,
    }


per_subj_binned = []
for sid in ok_subjects[:20]:
    res = _per_subject_binned_phase(sid)
    if res is not None:
        per_subj_binned.append(res)

if len(per_subj_binned) >= 1:
    centers = per_subj_binned[0]['bin_centers']
    grp = np.stack([r['mean'] for r in per_subj_binned], axis=0)  # (n_subj, n_bins)
    grp_mean = np.nanmean(grp, axis=0)
    grp_sem = np.nanstd(grp, axis=0, ddof=1) / np.sqrt(np.sum(~np.isnan(grp), axis=0))
    fig, ax = plt.subplots(figsize=(5.5, 4.0))
    for r in per_subj_binned:
        ax.plot(np.degrees(r['bin_centers']), r['mean'], color='lightgray',
                alpha=0.5, lw=0.7)
    ax.plot(np.degrees(centers), grp_mean, color='k', lw=2.0,
            label=f'group mean (n={len(per_subj_binned)})')
    ax.fill_between(np.degrees(centers),
                    grp_mean - grp_sem, grp_mean + grp_sem,
                    color='k', alpha=0.15, lw=0)
    ax.set_xlabel('aggregate ACC theta phase (deg)')
    ax.set_ylabel('mean Haufe-scaled reactivation (avg electrodes)')
    ax.legend(frameon=False, fontsize=8)
    sns.despine(ax=ax)
    fig.tight_layout()
    _save(fig, 'fig04_binned_phase_reactivation.png')
    plt.show()


### Pass 2 - real subject mirror (MS012)

Recompute PAC inline via `_walkthrough_real_subject` (loads cue HFA, applies the same
**positive-RPE trial filter** as the batch runner when enabled, builds **per-electrode**
Haufe-scaled reactivation `cue_hfa * w`, extracts ACC theta phase on the same grid, runs
`Pac(idpac=(2, 1, 0))` per electrode) so the swap-phase surrogate distribution is available
for Panel 5. The figure mirrors the synthetic walkthrough panel by panel; the example trial is
the hit trial with the highest absolute **aggregate** PAC z (mean over electrodes; fallback:
global max |z|). The highlighted electrode maximizes mean |per-electrode z| across trials
(fallback: largest |Haufe weight|).

Panel 6 swaps the synthetic `locked vs unlocked` overlay for the natural real-data
analogue, `hit vs miss`.


In [ ]:
# -----------------------------
# Pass 2: real subject methods walkthrough (per-electrode Tort MI PAC)
# -----------------------------


def _walkthrough_real_subject(subject_id: str,
                              n_perm: int = N_PAC_PERM,
                              random_state: int = PAC_RANDOM_STATE,
                              ) -> Dict[str, Any]:
    """Recompute ACC theta PAC for one subject with full surrogate capture."""
    subj = _build_subject(subject_id)
    acc_cue_hfa, cue_times, acc_hfa_labels = _load_acc_cue_hfa(subj)
    w_subj, kept_labels, kept_idx = subject_acc_haufe_vector(
        haufe_patterns_df, subject_id, SELECTED_CONTRAST, acc_hfa_labels,
        value_col=HAUFE_WEIGHT_COLUMN, min_kept=MIN_PATTERN_ELECTRODES,
    )
    if w_subj is None:
        raise RuntimeError(f'no Haufe match for {subject_id}')
    acc_cue_hfa = acc_cue_hfa[:, kept_idx, :]
    phase_mv, acc_phase_labels = _load_acc_theta_phase(subj, cue_times)
    common_labels, hfa_pos, pha_pos = np.intersect1d(
        kept_labels, acc_phase_labels, return_indices=True,
    )
    if common_labels.size < MIN_PATTERN_ELECTRODES:
        raise RuntimeError(f'too few common electrodes for {subject_id}')
    acc_cue_hfa = acc_cue_hfa[:, hfa_pos, :]
    phase_mv = phase_mv[:, pha_pos, :]
    kept_labels = common_labels.astype(str)
    w_subj = w_subj[hfa_pos]

    n_walk = int(acc_cue_hfa.shape[0])
    hit_map = _load_hit_miss_for_cue(subj)
    if len(hit_map) != n_walk:
        raise RuntimeError(
            f'epoch count mismatch walkthrough: behavior={len(hit_map)} hfa={n_walk}'
        )
    is_hit = (hit_map['hit_miss'].astype(str) == 'hit').to_numpy()
    is_miss = (hit_map['hit_miss'].astype(str) == 'miss').to_numpy()
    pos_m = _pos_rpe_epoch_mask(subj._combined_df, subj.subject, n_walk)
    if RESTRICT_TRIALS_TO_POS_RPE:
        if int(np.count_nonzero(pos_m)) == 0:
            raise RuntimeError(f'no positive-RPE trials for {subject_id}')
        acc_cue_hfa = acc_cue_hfa[pos_m]
        phase_mv = phase_mv[pos_m]
        is_hit = is_hit[pos_m]
        is_miss = is_miss[pos_m]
    reactivation_elec = compute_acc_reactivation_elec(acc_cue_hfa, w_subj)

    pac_out = compute_per_electrode_acc_pac(
        phase_mv=phase_mv,
        cue_hfa_3d=acc_cue_hfa,
        w_subj=w_subj,
        n_perm=int(n_perm),
        random_state=int(random_state),
        store_surr_distribution=True,
    )
    return {
        'subject': subject_id,
        'cue_times': cue_times,
        'phase_mv': phase_mv,
        'reactivation_elec': reactivation_elec,
        'kept_labels': kept_labels,
        'w_subj': w_subj,
        'is_hit': is_hit,
        'is_miss': is_miss,
        'pac': pac_out,
    }


example_subject = 'MS012' if 'MS012' in subjects_with_patterns else (
    subjects_with_patterns[0] if len(subjects_with_patterns) > 0 else None
)
if example_subject is None:
    print('No eligible subject available for Pass 2.')
else:
    print(f'Pass 2 example subject: {example_subject}')
    R = _walkthrough_real_subject(example_subject)
    phase_real = R['phase_mv']
    react_e = R['reactivation_elec']
    cue_times_real = R['cue_times']
    pac_real = R['pac']
    z_real = pac_real['trial_pac_z']
    z_elec = pac_real['trial_pac_z_elec']
    is_hit_real = R['is_hit']
    is_miss_real = R['is_miss']
    w_real = R['w_subj']
    kept_labels_real = R['kept_labels']

    hit_indices_real = np.where(is_hit_real)[0]
    if hit_indices_real.size > 0 and np.any(np.isfinite(z_real[hit_indices_real])):
        z_h = z_real[hit_indices_real]
        trial_pick = int(hit_indices_real[int(np.nanargmax(np.abs(z_h)))])
    else:
        trial_pick = int(np.nanargmax(np.abs(z_real)))

    mean_abs_z_e = np.nanmean(np.abs(z_elec), axis=0)
    if np.all(~np.isfinite(mean_abs_z_e)):
        elec_pick = int(np.argmax(np.abs(w_real)))
    else:
        elec_pick = int(np.nanargmax(mean_abs_z_e))

    print(f'  example trial : {trial_pick} '
          f'(z_agg = {z_real[trial_pick]:+.2f}, '
          f"{'hit' if is_hit_real[trial_pick] else ('miss' if is_miss_real[trial_pick] else 'other')})")
    print(f'  example elec  : {elec_pick} ({kept_labels_real[elec_pick]}, w={w_real[elec_pick]:+.3f})')

    lfp_proxy = np.cos(phase_real[trial_pick, elec_pick])

    phi_e = phase_real[trial_pick, elec_pick]
    r_e = react_e[trial_pick, elec_pick]

    fig = plt.figure(figsize=(13, 11))
    gs = fig.add_gridspec(3, 3, height_ratios=[1.0, 1.0, 1.0],
                          width_ratios=[1.0, 1.1, 1.0])

    ax1 = fig.add_subplot(gs[0, 0])
    ax1.plot(cue_times_real, lfp_proxy, lw=1.0, color='steelblue',
             label='cos(theta phase)')
    ax1.plot(cue_times_real, r_e, lw=1.3,
             color='firebrick', label='w·HFA (same elec)')
    ax1.set_xlabel('cue time (s)')
    ax1.set_ylabel('amplitude (a.u.)')
    ax1.set_title(
        f'Panel 1 - elec {kept_labels_real[elec_pick]} '
        f'(trial {trial_pick}, '
        f"{'hit' if is_hit_real[trial_pick] else 'miss'})"
    )
    ax1_t = ax1.twinx()
    ax1_t.plot(cue_times_real, phi_e,
               color='gray', alpha=0.6, lw=0.7, label='theta phase')
    ax1_t.set_ylim(-np.pi, np.pi)
    ax1_t.set_ylabel('theta phase (rad)')
    ax1.legend(loc='upper left', fontsize=7, frameon=False)
    ax1_t.legend(loc='upper right', fontsize=7, frameon=False)
    sns.despine(ax=ax1, right=False)

    ax2a = fig.add_subplot(gs[0, 1])
    ax2b = fig.add_subplot(gs[1, 1])
    extent = [cue_times_real[0], cue_times_real[-1],
              phase_real.shape[1] - 0.5, -0.5]
    im_ph = ax2a.imshow(phase_real[trial_pick], aspect='auto', cmap='twilight',
                        vmin=-np.pi, vmax=np.pi, extent=extent)
    ax2a.set_title(f'Panel 2a - ACC theta phase (trial {trial_pick})')
    ax2a.set_ylabel('electrode')
    ax2a.set_xlabel('time (s)')
    plt.colorbar(im_ph, ax=ax2a, fraction=0.04, pad=0.02, label='phase (rad)')
    rmax = float(np.nanpercentile(np.abs(react_e[trial_pick]), 99)) or 1.0
    im_r = ax2b.imshow(react_e[trial_pick], aspect='auto', cmap='magma',
                       extent=extent, vmin=-rmax, vmax=rmax)
    ax2b.set_title('Panel 2b - Haufe-scaled cue HFA (per electrode)')
    ax2b.set_ylabel('electrode')
    ax2b.set_xlabel('time (s)')
    plt.colorbar(im_r, ax=ax2b, fraction=0.04, pad=0.02, label='w·HFA')

    ax3 = fig.add_subplot(gs[1, 0])
    sc = ax3.scatter(phi_e, r_e, c=cue_times_real,
                     s=6, cmap='viridis', alpha=0.6)
    ax3.set_xlim(-np.pi, np.pi)
    ax3.set_xlabel('theta phase (rad, same elec)')
    ax3.set_ylabel('Haufe-scaled reactivation (same elec)')
    plt.colorbar(sc, ax=ax3, fraction=0.04, pad=0.02, label='time (s)')
    sns.despine(ax=ax3)

    n_bins = 18
    edges = np.linspace(-np.pi, np.pi, n_bins + 1)
    centers = 0.5 * (edges[:-1] + edges[1:])
    bin_idx = np.clip(np.digitize(phi_e, edges) - 1, 0, n_bins - 1)
    bin_mean_real = np.array([
        float(np.nanmean(r_e[bin_idx == b]))
        if int((bin_idx == b).sum()) > 0 else np.nan
        for b in range(n_bins)
    ])
    ax4 = fig.add_subplot(gs[0, 2], projection='polar')
    ax4.bar(centers, bin_mean_real - np.nanmin(bin_mean_real),
            width=np.diff(edges), align='center', alpha=0.7,
            color='firebrick', edgecolor='k', linewidth=0.4)
    ax4.set_title(f'Panel 4 - polar (trial {trial_pick}, elec {elec_pick})', y=1.08)
    ax4.set_yticklabels([])

    ax5 = fig.add_subplot(gs[2, 0])
    surr_e = pac_real['surr_distribution_elec']
    if surr_e is not None:
        surr_t = surr_e[elec_pick, :, trial_pick]
        ax5.hist(surr_t, bins=30, color='gray', alpha=0.7,
                 edgecolor='black', linewidth=0.4, label='swap-phase surrogates')
        obs = pac_real['pac_raw_elec'][trial_pick, elec_pick]
        ax5.axvline(obs, color='firebrick', lw=2.0, label='observed PAC')
        z_line = pac_real['trial_pac_z_elec'][trial_pick, elec_pick]
        ax5.set_title(
            f'Panel 5 - null vs observed (trial {trial_pick}, elec {elec_pick})\n'
            f'z_e = {z_line:+.2f} | z_agg = {z_real[trial_pick]:+.2f}'
        )
        ax5.set_xlabel('raw Tort MI PAC')
        ax5.set_ylabel('surrogate count')
        ax5.legend(fontsize=7, frameon=False)
    sns.despine(ax=ax5)

    ax6 = fig.add_subplot(gs[2, 1:])
    z_hit = z_real[is_hit_real]
    z_miss = z_real[is_miss_real]
    if z_hit.size > 1:
        sns.kdeplot(z_hit, ax=ax6, label=f'hit (n={int(is_hit_real.sum())})',
                    fill=True, alpha=0.45, color='firebrick')
    if z_miss.size > 1:
        sns.kdeplot(z_miss, ax=ax6, label=f'miss (n={int(is_miss_real.sum())})',
                    fill=True, alpha=0.45, color='steelblue')
    ax6.axvline(0.0, color='k', lw=0.5, ls='--')
    if z_hit.size > 0:
        ax6.axvline(float(np.nanmean(z_hit)), color='firebrick', lw=1.0, ls=':')
    if z_miss.size > 0:
        ax6.axvline(float(np.nanmean(z_miss)), color='steelblue', lw=1.0, ls=':')
    ax6.set_xlabel('per-trial PAC z (mean over electrodes)')
    ax6.set_title(
        f'Panel 6 - per-trial z (hit mean={np.nanmean(z_hit) if z_hit.size else np.nan:+.3f}, '
        f'miss mean={np.nanmean(z_miss) if z_miss.size else np.nan:+.3f})'
    )
    ax6.legend(frameon=False, fontsize=8)
    sns.despine(ax=ax6)

    fig.suptitle(
        f'Pass 2: {example_subject}  -  per-electrode Tort MI PAC + Haufe-scaled HFA',
        fontsize=12, y=1.0,
    )
    fig.tight_layout()
    real_fig_path = figure_output_root / f'walkthrough_real_{example_subject}.png'
    fig.savefig(real_fig_path, dpi=160, bbox_inches='tight')
    print('saved', real_fig_path)
    plt.show()
